In [2]:
landing_path = (
    "abfss://Fleet_Logistics_Engineering@onelake.dfs.fabric.microsoft.com/"
    "Fleet_Logistics_Lakehouse.Lakehouse/Files/Landing/maintenance_records.csv"
)

StatementMeta(, c42c979f-5fb8-4a1f-86f6-19ab26289ec9, 4, Finished, Available, Finished, False)

In [3]:
df_maintenance = (
    spark.read
    .option("header", "true")
    .csv(landing_path)
)

display(df_maintenance)

df_maintenance.printSchema()

print(f"Source records: {df_maintenance.count()}")

StatementMeta(, c42c979f-5fb8-4a1f-86f6-19ab26289ec9, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a30aaaaa-3cae-4844-a6dc-d663f493099e)

root
 |-- maintenance_id: string (nullable = true)
 |-- truck_id: string (nullable = true)
 |-- maintenance_date: string (nullable = true)
 |-- maintenance_type: string (nullable = true)
 |-- odometer_reading: string (nullable = true)
 |-- labor_hours: string (nullable = true)
 |-- labor_cost: string (nullable = true)
 |-- parts_cost: string (nullable = true)
 |-- total_cost: string (nullable = true)
 |-- facility_location: string (nullable = true)
 |-- downtime_hours: string (nullable = true)
 |-- service_description: string (nullable = true)

Source records: 2920


In [4]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType,
    DateType
)

# Maintenance_Records — Explicit Schema

maintenance_schema = StructType([
    StructField("maintenance_id", StringType(), True),
    StructField("truck_id", StringType(), True),
    StructField("maintenance_date", DateType(), True),
    StructField("maintenance_type", StringType(), True),
    StructField("odometer_reading", IntegerType(), True),
    StructField("labor_hours", DoubleType(), True),
    StructField("labor_cost", DoubleType(), True),
    StructField("parts_cost", DoubleType(), True),
    StructField("total_cost", DoubleType(), True),
    StructField("facility_location", StringType(), True),
    StructField("downtime_hours", DoubleType(), True),
    StructField("service_description", StringType(), True)
])

StatementMeta(, c42c979f-5fb8-4a1f-86f6-19ab26289ec9, 6, Finished, Available, Finished, False)

In [5]:
landing_path = (
    "abfss://Fleet_Logistics_Engineering@onelake.dfs.fabric.microsoft.com/"
    "Fleet_Logistics_Lakehouse.Lakehouse/Files/Landing/maintenance_records.csv"
)

df_maintenance = (
    spark.read
    .option("header", "true")
    .schema(maintenance_schema)
    .csv(landing_path)
)

display(df_maintenance.limit(20))

df_maintenance.printSchema()

print(f"Source records: {df_maintenance.count()}")

StatementMeta(, c42c979f-5fb8-4a1f-86f6-19ab26289ec9, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 38ed3e62-5c11-47c4-8d93-4d21c0c8c790)

root
 |-- maintenance_id: string (nullable = true)
 |-- truck_id: string (nullable = true)
 |-- maintenance_date: date (nullable = true)
 |-- maintenance_type: string (nullable = true)
 |-- odometer_reading: integer (nullable = true)
 |-- labor_hours: double (nullable = true)
 |-- labor_cost: double (nullable = true)
 |-- parts_cost: double (nullable = true)
 |-- total_cost: double (nullable = true)
 |-- facility_location: string (nullable = true)
 |-- downtime_hours: double (nullable = true)
 |-- service_description: string (nullable = true)

Source records: 2920


In [6]:
# 1. NULL primary key
null_maintenance_ids = (
    df_maintenance
    .filter(F.col("maintenance_id").isNull())
    .count()
)

print(f"NULL maintenance IDs: {null_maintenance_ids}")

StatementMeta(, c42c979f-5fb8-4a1f-86f6-19ab26289ec9, 8, Finished, Available, Finished, False)

NULL maintenance IDs: 0


In [7]:
# 2. Duplicate primary key
duplicate_maintenance_ids = (
    df_maintenance
    .groupBy("maintenance_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"Duplicate maintenance IDs: {duplicate_maintenance_ids}")

StatementMeta(, c42c979f-5fb8-4a1f-86f6-19ab26289ec9, 9, Finished, Available, Finished, False)

Duplicate maintenance IDs: 0


In [8]:
# 3. NULL maintenance dates
null_maintenance_dates = (
    df_maintenance
    .filter(F.col("maintenance_date").isNull())
    .count()
)

print(f"NULL maintenance dates: {null_maintenance_dates}")

StatementMeta(, c42c979f-5fb8-4a1f-86f6-19ab26289ec9, 10, Finished, Available, Finished, False)

NULL maintenance dates: 0


In [9]:
# 5. Invalid labor hours
invalid_labor_hours = (
    df_maintenance
    .filter(
        F.col("labor_hours").isNull() |
        (F.col("labor_hours") < 0)
    )
    .count()
)

print(f"Invalid labor hours: {invalid_labor_hours}")

StatementMeta(, c42c979f-5fb8-4a1f-86f6-19ab26289ec9, 11, Finished, Available, Finished, False)

Invalid labor hours: 0


In [10]:
# 4. Invalid odometer readings
invalid_odometer = (
    df_maintenance
    .filter(
        F.col("odometer_reading").isNull() |
        (F.col("odometer_reading") < 0)
    )
    .count()
)

print(f"Invalid odometer readings: {invalid_odometer}")

StatementMeta(, c42c979f-5fb8-4a1f-86f6-19ab26289ec9, 12, Finished, Available, Finished, False)

Invalid odometer readings: 0


In [11]:
# 6. Invalid labor cost
invalid_labor_cost = (
    df_maintenance
    .filter(
        F.col("labor_cost").isNull() |
        (F.col("labor_cost") < 0)
    )
    .count()
)

print(f"Invalid labor cost: {invalid_labor_cost}")

StatementMeta(, c42c979f-5fb8-4a1f-86f6-19ab26289ec9, 13, Finished, Available, Finished, False)

Invalid labor cost: 0


In [12]:
# 7. Invalid parts cost
invalid_parts_cost = (
    df_maintenance
    .filter(
        F.col("parts_cost").isNull() |
        (F.col("parts_cost") < 0)
    )
    .count()
)

print(f"Invalid parts cost: {invalid_parts_cost}")

StatementMeta(, c42c979f-5fb8-4a1f-86f6-19ab26289ec9, 14, Finished, Available, Finished, False)

Invalid parts cost: 0


In [13]:
# 8. Invalid total cost
invalid_total_cost = (
    df_maintenance
    .filter(
        F.col("total_cost").isNull() |
        (F.col("total_cost") < 0)
    )
    .count()
)

print(f"Invalid total cost: {invalid_total_cost}")

StatementMeta(, c42c979f-5fb8-4a1f-86f6-19ab26289ec9, 15, Finished, Available, Finished, False)

Invalid total cost: 0


In [14]:
# 9. Invalid downtime
invalid_downtime = (
    df_maintenance
    .filter(
        F.col("downtime_hours").isNull() |
        (F.col("downtime_hours") < 0)
    )
    .count()
)

print(f"Invalid downtime hours: {invalid_downtime}")

StatementMeta(, c42c979f-5fb8-4a1f-86f6-19ab26289ec9, 16, Finished, Available, Finished, False)

Invalid downtime hours: 0


In [15]:
# Refrential Integrity
invalid_maintenance_truck_ids = (
    df_maintenance
    .filter(F.col("truck_id").isNotNull())
    .join(
        spark.table("bronze_trucks").select("truck_id"),
        on="truck_id",
        how="left_anti"
    )
    .count()
)

print(
    f"Invalid non-NULL maintenance truck IDs: "
    f"{invalid_maintenance_truck_ids}"
)

StatementMeta(, c42c979f-5fb8-4a1f-86f6-19ab26289ec9, 17, Finished, Available, Finished, False)

Invalid non-NULL maintenance truck IDs: 0


In [16]:
#Final Validation
if null_maintenance_ids > 0:
    raise ValueError(
        "ETL failed: NULL maintenance_id values detected."
    )

if duplicate_maintenance_ids > 0:
    raise ValueError(
        "ETL failed: Duplicate maintenance_id values detected."
    )

if null_maintenance_dates > 0:
    raise ValueError(
        "ETL failed: NULL maintenance_date values detected."
    )

if invalid_odometer > 0:
    raise ValueError(
        "ETL failed: Invalid odometer readings detected."
    )

if invalid_labor_hours > 0:
    raise ValueError(
        "ETL failed: Invalid labor hours detected."
    )

if invalid_labor_cost > 0:
    raise ValueError(
        "ETL failed: Invalid labor cost detected."
    )

if invalid_parts_cost > 0:
    raise ValueError(
        "ETL failed: Invalid parts cost detected."
    )

if invalid_total_cost > 0:
    raise ValueError(
        "ETL failed: Invalid total cost detected."
    )

if invalid_downtime > 0:
    raise ValueError(
        "ETL failed: Invalid downtime hours detected."
    )

if invalid_maintenance_truck_ids > 0:
    raise ValueError(
        "ETL failed: Maintenance records contain "
        "truck IDs not found in bronze_trucks."
    )

print("Maintenance data quality and referential-integrity validation passed.")

StatementMeta(, c42c979f-5fb8-4a1f-86f6-19ab26289ec9, 18, Finished, Available, Finished, False)

Maintenance data quality and referential-integrity validation passed.


In [17]:
df_maintenance_bronze = (
    df_maintenance
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("source_file", F.lit("maintenance_records.csv"))
)

display(df_maintenance_bronze.limit(10))

StatementMeta(, c42c979f-5fb8-4a1f-86f6-19ab26289ec9, 19, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, afe0ad76-27c5-4c49-b3a8-bff14cc581e4)

In [18]:
df_maintenance_bronze.createOrReplaceTempView(
    "maintenance_records_source"
)

StatementMeta(, c42c979f-5fb8-4a1f-86f6-19ab26289ec9, 20, Finished, Available, Finished, False)

In [19]:
spark.sql("""
CREATE TABLE IF NOT EXISTS bronze_maintenance_records (
    maintenance_id STRING,
    truck_id STRING,
    maintenance_date DATE,
    maintenance_type STRING,
    odometer_reading INT,
    labor_hours DOUBLE,
    labor_cost DOUBLE,
    parts_cost DOUBLE,
    total_cost DOUBLE,
    facility_location STRING,
    downtime_hours DOUBLE,
    service_description STRING,
    ingestion_timestamp TIMESTAMP,
    source_file STRING
)
""")

print("bronze_maintenance_records table is ready.")

StatementMeta(, c42c979f-5fb8-4a1f-86f6-19ab26289ec9, 21, Finished, Available, Finished, False)

bronze_maintenance_records table is ready.


In [20]:
spark.sql("""
MERGE INTO bronze_maintenance_records AS target

USING maintenance_records_source AS source

ON target.maintenance_id = source.maintenance_id

WHEN MATCHED THEN
    UPDATE SET
        target.truck_id = source.truck_id,
        target.maintenance_date = source.maintenance_date,
        target.maintenance_type = source.maintenance_type,
        target.odometer_reading = source.odometer_reading,
        target.labor_hours = source.labor_hours,
        target.labor_cost = source.labor_cost,
        target.parts_cost = source.parts_cost,
        target.total_cost = source.total_cost,
        target.facility_location = source.facility_location,
        target.downtime_hours = source.downtime_hours,
        target.service_description = source.service_description,
        target.ingestion_timestamp = source.ingestion_timestamp,
        target.source_file = source.source_file

WHEN NOT MATCHED THEN
    INSERT (
        maintenance_id,
        truck_id,
        maintenance_date,
        maintenance_type,
        odometer_reading,
        labor_hours,
        labor_cost,
        parts_cost,
        total_cost,
        facility_location,
        downtime_hours,
        service_description,
        ingestion_timestamp,
        source_file
    )

    VALUES (
        source.maintenance_id,
        source.truck_id,
        source.maintenance_date,
        source.maintenance_type,
        source.odometer_reading,
        source.labor_hours,
        source.labor_cost,
        source.parts_cost,
        source.total_cost,
        source.facility_location,
        source.downtime_hours,
        source.service_description,
        source.ingestion_timestamp,
        source.source_file
    )
""")

print("Maintenance Bronze MERGE completed successfully.")

StatementMeta(, c42c979f-5fb8-4a1f-86f6-19ab26289ec9, 22, Finished, Available, Finished, False)

Maintenance Bronze MERGE completed successfully.


In [21]:
bronze_maintenance_count = (
    spark.table("bronze_maintenance_records")
    .count()
)

print(
    f"Bronze maintenance records: "
    f"{bronze_maintenance_count}"
)

StatementMeta(, c42c979f-5fb8-4a1f-86f6-19ab26289ec9, 23, Finished, Available, Finished, False)

Bronze maintenance records: 2920
